# FlowGuard experiment walkthrough

Run this notebook from the project root using the environment in README.md. The included models and results come from recorded network flows. Cells that retrain models replace existing experiment artifacts.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
root = Path.cwd()
assert (root / "nids").exists(), "Open this notebook from the project root"
audit = json.loads((root / "data/processed/data_audit.json").read_text())
display(pd.DataFrame(audit["splits"]))

## Inspect the actual experiment evidence
The random split excludes repeated selected-feature hashes, but it does not isolate related sessions. Compare per-class results as well as accuracy.

In [ ]:
display(pd.read_csv(root / "results/model_comparison.csv"))
display(Image(filename=str(root / "results/plots/Transformer_confusion.png")))

## Classify held-out flow examples
Ground-truth labels live in a separate file. Prediction only consumes the 24 feature columns.

In [ ]:
from nids.inference import Predictor
flows = pd.read_csv(root / "data/samples/demo_flows.csv")
model = Predictor(root / "artifacts")
result = model.predict(flows)
truth = pd.read_csv(root / "data/samples/demo_flows_labeled.csv")["target"]
result["actual_class"] = truth
display(result)

## Inspect a local sensitivity probe
This replaces individual features with training medians and reports changes in the predicted-class probability. It is not SHAP/LIME or causal attribution.

In [ ]:
display(model.occlusion_explanation(flows.iloc[:1]))

## Optional: retrain the fixed experiment
Execute this only when you intend to replace saved checkpoints and results. Hyperparameters and selected epochs never use test-set feedback.

In [ ]:
# Uncomment to retrain using the same Python environment
# import subprocess, sys
# subprocess.run([sys.executable, "scripts/train.py", "--epochs", "18", "--patience", "5", "--seed", "42"], check=True)

## Start the dashboard
In a terminal, run `python -m streamlit run app.py`. Read docs/PROJECT_REPORT.md and docs/DEMO_AND_VIVA.md before presentation.

## Extended project experiments
The cells above describe the original five-class feature benchmark. The current package also includes seven-class classification, chronological attention, optional architectures and adaptation. Physical capture still requires Npcap.


In [ ]:
pd.read_csv("results/extended/model_comparison.csv")


In [ ]:
pd.read_csv("results/advanced/model_comparison.csv")


In [ ]:
from nids.advanced_inference import AdvancedPredictor
advanced = AdvancedPredictor("artifacts/advanced", "TemporalTransformer")
advanced.predict(pd.read_csv("data/temporal/test_flows.csv")).head()
